In [ ]:
from rdflib import Graph
from rdfine import GraphReader
from compilers import (
    PipelineGenerator,
    ProjectBuilder,
    LdioConfigCompiler)

#### Loading the graph

In [2]:
# Loading the graph
input_folder = "..\\data\\"
catalog_graph = Graph()
catalog_graph.parse(input_folder + "catalog.ttl", publicID = "file:///workspace/pipeline/")
catalog_reader = GraphReader(catalog_graph)
catalog_reader = catalog_reader.infer(input_folder + "inference_rules.yaml")

#### Compiling the pipeline build

`PipelineGenerator` orchestrates the full chain. It runs `PipelineExtractor` first (the only compiler that needs the `pipeline_id`), then walks the `Compiler._registry` in tier order, invoking every compiler whose `applies_to` returns `True` against the growing build graph.

In [ ]:
pipeline_id = ":DemonstratorPipeline"
gen = PipelineGenerator(pipeline_id, catalog_reader.graph)
build_graph = gen.compile()

# Which compilers actually ran?
[cls.__name__ for cls in gen.compilers]

#### Inspecting the compiled files

Every FILE-tier compiler attaches a `tcs:File` node to the `tcs:PipelineBuild` via `tcs:compiledFile`. The build graph is now self-describing: it knows which files should be written, where, and with what content.

`ProjectBuilder` collects those nodes into a DataFrame on `builder.files` for inspection before any IO happens.

In [ ]:
builder = ProjectBuilder(build_graph)

for _, row in builder.files.iterrows():
    print(f"=== {row['filepath']}/{row['filename']} ===")
    print(row['content'])
    print()

#### Writing the project to disk

`ProjectBuilder.write(target_dir)` materializes every collected file under the given directory, creating parent folders as needed. Existing files at the same path are overwritten. The call returns the absolute paths it wrote.

In [ ]:
written = builder.write("../out/demonstrator")
for path in written:
    print(path)

#### Inspecting compiler internals

`PipelineGenerator` keeps the compiler instances it ran on `gen.compilers`, keyed by class. Each instance retains its intermediate state — useful for debugging when an output doesn't look right.

In [ ]:
gen.compilers[LdioConfigCompiler].df_steps